## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. Download the dataset first: click [Airplane Crashes and Fatalities upto 2023.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/W4%20Gen%20AI/W4D3/Airplane%20Crashes%20and%20Fatalities%20upto%202023.zip). It downloads `Airplane Crashes and Fatalities upto 2023.zip`. Save it in `C:\Users\sivan\Datasets`.
3. When the upload cell asks for a file, choose `Airplane Crashes and Fatalities upto 2023.zip` from `C:\Users\sivan\Datasets`.

**Run the cells from top to bottom** (Shift+Enter runs one cell). After a restart, run them again from the top.

# Daily Challenge: Analysis of Airplane Crashes and Fatalities

## 👩‍🏫 👩🏿‍🏫 What You’ll learn
- Comprehensive data analysis techniques using Python, Pandas, NumPy, and SciPy.
- Methods for data cleaning, exploratory analysis, statistical testing, and visualization.
- Insightful interpretation of complex datasets.

## 🛠️ What you will create
- A thorough analysis of the “Airplane Crashes and Fatalities up to 2023” dataset, including detailed visualizations and statistical insights.

## Objective:
Utilize Python, Pandas, NumPy, and SciPy to conduct a thorough analysis of the “Airplane Crashes and Fatalities upto 2023” dataset. This challenge will encompass data cleaning, exploratory analysis, statistical testing, and visualization to draw meaningful insights.

## Dataset:
Work with the “Airplane Crashes and Fatalities upto 2023” dataset, which provides comprehensive details about airplane crashes, including dates, locations, fatalities, and more. Access the dataset [here](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/W4%20Gen%20AI/W4D3/Airplane%20Crashes%20and%20Fatalities%20upto%202023.zip).

## Tasks:
**1. Data Import and Cleaning:**
- Import the dataset using Pandas.
- Clean and preprocess the data, addressing missing values and categorizing data as needed.
- Convert dates and other relevant fields to appropriate formats.

**2. Exploratory Data Analysis:**
- Use Pandas to explore basic statistics such as the number of crashes, fatalities, and survival rates.
- Analyze the frequency of crashes over time to identify any trends.

**3. Statistical Analysis:**
- Apply SciPy to analyze the distribution of fatalities and survival rates. Calculate key statistics like mean, median, and standard deviation.
- Conduct a hypothesis test (e.g., comparing the average number of fatalities in different decades or regions).

**4. Visualization:**
- Create charts and graphs using Matplotlib and Seaborn to visualize the findings from your exploratory data analysis and statistical tests.
- Examples might include time series plots of crashes over years, bar charts of crashes by region, and histograms of fatalities.

**5. Insight and Report:**
- Summarize your findings and provide insights into the patterns or anomalies discovered in the data.
- Prepare a well-structured report including all code, visualizations, and interpretations.

## Submission:
You should submit a comprehensive report encompassing their code, analysis, and visualizations. Articulate how you have applied NumPy, Pandas, and SciPy in your analysis, highlighting any significant patterns or insights discovered in the dataset.

## Setup
Run this cell first. It imports the libraries used in the analysis.

In [ ]:
import pandas as pd
# pandas: for working with tables (DataFrames)
import numpy as np
# numpy: for math on whole columns
import matplotlib.pyplot as plt
# matplotlib: for drawing charts
import seaborn as sns
# seaborn: for nicer statistical charts
from scipy import stats
# stats: the statistics part of SciPy

---
# 1. Data Import and Cleaning

In [ ]:
from google.colab import files
# the Colab tool for uploading files from your computer
uploaded = files.upload()
# choose "Airplane Crashes and Fatalities upto 2023.zip" (saved in a variable so its contents are not printed)
!unzip "Airplane Crashes and Fatalities upto 2023.zip"
# unzip it: you get the file "Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv"

In [ ]:
crashes = pd.read_csv('Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv', encoding='latin-1')
# load the CSV; encoding='latin-1' is needed because the file has special letters (é, ü ...) that the default reader cannot read
print(crashes.shape)
# (number of rows, number of columns)
print(crashes.head(3).T)
# the first 3 crashes, turned sideways (.T) so all 17 columns fit on the screen

In [ ]:
crashes.info()
# column names, types and how many values are not empty

In [ ]:
missing = crashes.isnull().sum()
# count the empty cells in each column
print(missing[missing > 0])
# show only the columns that have empty cells
print('Duplicate rows:', crashes.duplicated().sum())
# rows that are exact copies

**What we have:** **4,998 crashes** (rows) and **17 columns**, from 1908 to 2023. There are no duplicate rows.
- **Numbers** (decimal type): Aboard, Fatalities and Ground, each also split into passengers and crew. ("Passangers" is spelled wrong in the file; we keep the original names.)
- **Text:** Date (stored as text such as "9/17/1908"), Time, Location, Operator, Route, aircraft type (AC Type), Registration and a written Summary.
- **Missing values:** Time is missing in 1,512 rows and Flight # in 3,669, but these do not matter for our questions. The important ones are few: **Aboard is missing in 18 rows, Fatalities in 8, Ground in 42, Operator in 10, Location in 4**.
- **Cleaning plan:**
  - Convert Date to a real date and add Year and Decade.
  - Fill the missing text values with "Unknown".
  - Treat missing Ground as 0.
  - Remove the 18 rows without Aboard/Fatalities (we cannot compute a survival rate for them).

In [ ]:
crashes['Date'] = pd.to_datetime(crashes['Date'], format='%m/%d/%Y')
# turn the text dates (month/day/year) into real dates
crashes['Year'] = crashes['Date'].dt.year
# the year of each crash
crashes['Decade'] = (crashes['Year'] // 10) * 10
# the decade: 1987 // 10 = 198, * 10 = 1980
print(crashes['Date'].min(), '→', crashes['Date'].max())
# the first and the last crash in the data

In [ ]:
for col in ['Operator', 'Location', 'AC Type', 'Route']:
    # text columns where an empty cell just means "not known"
    crashes[col] = crashes[col].fillna('Unknown')
    # write 'Unknown' instead of leaving the cell empty
crashes['Ground'] = crashes['Ground'].fillna(0)
# people killed on the ground: an empty cell almost always means nobody (0)
before = len(crashes)
# number of rows before removing anything
crashes = crashes.dropna(subset=['Aboard', 'Fatalities'])
# remove the few crashes where the number of people aboard or killed is unknown
print('Rows removed:', before - len(crashes), '| rows left:', len(crashes))
# how many rows we dropped and how many remain

In [ ]:
crashes['Survivors'] = crashes['Aboard'] - crashes['Fatalities']
# people aboard who survived
crashes['Survival rate'] = np.where(crashes['Aboard'] > 0, crashes['Survivors'] / crashes['Aboard'], np.nan)
# share of the people aboard who survived (0 = nobody, 1 = everybody); empty if nobody was aboard
crashes['Type'] = np.where(crashes['Operator'].str.contains('Military', case=False), 'Military', 'Civil')
# Military if the operator name contains "Military", otherwise Civil

In [ ]:
us_states = ['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware',
             'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky',
             'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
             'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico',
             'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
             'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont',
             'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming', 'DC', 'Washington DC']
# the 50 US states (plus DC): in this file US crashes are written "City, State" instead of "City, Country"
uk_parts = ['England', 'Scotland', 'Wales', 'Northern Ireland']
# parts of the United Kingdom, written separately in the file
crashes['Country'] = crashes['Location'].str.split(',').str[-1].str.strip()
# the country is the last part after the last comma ("Moscow, Russia" → "Russia")
crashes['Country'] = crashes['Country'].replace(dict.fromkeys(us_states, 'USA'))
# a US state becomes "USA"
crashes['Country'] = crashes['Country'].replace(dict.fromkeys(uk_parts, 'United Kingdom'))
# England, Scotland ... become "United Kingdom"
crashes['Country'] = crashes['Country'].replace({'USSR': 'Russia'})
# the old Soviet Union is counted together with Russia
print('Different countries:', crashes['Country'].nunique())
# how many countries are left
print(crashes['Country'].value_counts().head(10))
# the 10 countries with the most crashes

**What the cleaning did:**
- **Dates** were converted with the format month/day/year with no failures: the data runs from **17 Sep 1908 to 1 Nov 2023**.
- **18 rows were removed** (the 8 rows without Fatalities are among the 18 without Aboard), leaving **4,980 crashes**.
- **New columns:**
  - **Year** and **Decade**.
  - **Survivors** = Aboard − Fatalities.
  - **Survival rate** = Survivors / Aboard (left empty for the 5 crashes with nobody aboard).
  - **Type:** **752 Military** and **4,228 Civil** crashes. Military means the operator name contains "Military", such as "Military - U.S. Air Force".
  - **Country:** the last part of Location. US crashes are written "City, State", so the states were changed to **USA**. England, Scotland, Wales and Northern Ireland became United Kingdom, and USSR was joined with Russia.
- **Top countries:** **USA 1,005 crashes**, then Russia 324, Brazil 175, Colombia 152, UK 133, Canada 125 and France 124.
- **Limitation:** there are still 458 "countries", because many locations are written differently: oceans ("Atlantic Ocean, 110 miles West of Ireland"), "Near …" text, or spelling variants. Also, "Georgia" was treated as the US state, although a few crashes may be in the country Georgia. The top of the list is reliable; the long tail is messy.

---
# 2. Exploratory Data Analysis

In [ ]:
total_aboard = crashes['Aboard'].sum()
# all the people who were on the crashed planes
total_dead = crashes['Fatalities'].sum()
# all the people killed aboard
print('Number of crashes:', len(crashes))
# one row = one crash
print('People aboard:', int(total_aboard))
# total people aboard
print('Fatalities aboard:', int(total_dead))
# total people killed aboard
print('Killed on the ground:', int(crashes['Ground'].sum()))
# people killed on the ground
print('Overall survival rate:', round(1 - total_dead / total_aboard, 3))
# share of all people aboard who survived
print('Crashes with no survivors:', (crashes['Survivors'] == 0).sum())
# crashes where everybody aboard died
print(crashes[['Aboard', 'Fatalities', 'Ground', 'Survival rate']].describe().round(2))
# basic statistics of the main number columns

In [ ]:
print(crashes['Type'].value_counts())
# number of military and civil crashes
print(crashes.groupby('Type')[['Aboard', 'Fatalities', 'Survival rate']].mean().round(2))
# average people aboard, killed and survival rate for each type
print(crashes['Operator'].value_counts().head(10))
# the 10 operators with the most crashes
print(crashes['AC Type'].value_counts().head(5))
# the 5 aircraft types with the most crashes

In [ ]:
print(crashes.nlargest(5, 'Fatalities')[['Date', 'Location', 'Operator', 'Aboard', 'Fatalities', 'Ground']].to_string(index=False))
# the 5 crashes with the most deaths aboard

In [ ]:
per_year = crashes.groupby('Year').size()
# number of crashes in each year
print('Worst years:')
# title
print(per_year.nlargest(5))
# the 5 years with the most crashes
decade_table = crashes.groupby('Decade').agg(crashes=('Date', 'size'), fatalities=('Fatalities', 'sum'),
                                            mean_fatalities=('Fatalities', 'mean'), survival=('Survival rate', 'mean'))
# for each decade: number of crashes, total deaths, average deaths per crash, average survival rate
print(decade_table.round(2))
# show the table

In [ ]:
years = per_year.index.to_numpy()
# the years as a NumPy array
counts = per_year.to_numpy()
# the crash counts as a NumPy array
since_1970 = years >= 1970
# True for the years from 1970 on
r, p = stats.pearsonr(years[since_1970], counts[since_1970])
# correlation between year and number of crashes since 1970
fit = stats.linregress(years[since_1970], counts[since_1970])
# a straight trend line through the yearly counts since 1970
print('Since 1970: Pearson r =', round(r, 3), '| p =', f'{p:.2e}')
# negative r = fewer crashes as the years go by
print('Trend: about', round(fit.slope, 2), 'crashes per year, every year')
# how many crashes per year are added or removed each year

**What we see:**
- **4,980 crashes**, with **155,356 people aboard** and **111,561 killed aboard**, plus 8,510 people killed on the ground. (The 9/11 ground victims, 2,750, are written on both New York planes, so the ground total counts them twice.)
- **Survival:** of everyone aboard, **28.2% survived**. But in a typical crash the picture is worse: in **3,182 crashes (64%) nobody survived**. The average survival rate per crash is only 0.18, because big crashes with survivors (large airliners) raise the overall rate while most crashes are small planes where everyone died.
- **Military vs civil:** military crashes kill more people on average (**26.5 vs 21.7**) and have a much lower survival rate (**6.5% vs 20.2%**).
- **Operators and aircraft:** **Aeroflot** (Soviet/Russian airline, 255 crashes) and the U.S. Air Force (140) have the most crashes. The **Douglas DC-3** is the most common type (330 crashes, plus 133 of its military version C-47/C-47A), because it was the most widely flown plane for decades.
- **Worst crash:** **Tenerife 1977** (two Boeing 747s collided on the runway), with **583 killed** of 644 aboard. Next comes Japan Air Lines 1985, with 520 killed.
- **Trend over time:**
  - Crashes rose fast until the **late 1940s**. The worst year is **1946 (87 crashes)**, after World War II, when many old military planes were used.
  - From about 1950 to 2000 there are 50–80 crashes every year.
  - After 2000 the number **drops steadily to only 4 in 2023**. Since 1970 the correlation between year and number of crashes is **r = −0.772** (p ≈ 8e-12), about **one crash fewer every year**.
- **Per decade:** the deaths per crash grew from 5.7 (1930s) to **32.3 (1970s)** as planes got bigger, then fell to about 22–27.

---
# 3. Statistical Analysis

### Distribution of fatalities and survival rates (SciPy)

In [ ]:
survival = crashes['Survival rate'].dropna()
# survival rates (skip the crashes with nobody aboard)
dist = pd.DataFrame({
    'Fatalities': [stats.tmean(crashes['Fatalities']), np.median(crashes['Fatalities']), stats.tstd(crashes['Fatalities']),
                   stats.skew(crashes['Fatalities']), stats.kurtosis(crashes['Fatalities'])],
    'Survival rate': [stats.tmean(survival), np.median(survival), stats.tstd(survival),
                      stats.skew(survival), stats.kurtosis(survival)]},
    index=['mean', 'median', 'std', 'skewness', 'kurtosis'])
# mean (tmean), median, standard deviation (tstd), skewness and kurtosis of both columns
print(dist.round(3))
# show the table
print('Share of crashes where everyone died:', round((survival == 0).mean(), 3))
# survival rate exactly 0
print('Share of crashes where everyone survived:', round((survival == 1).mean(), 3))
# survival rate exactly 1

**The distributions:**
- **Fatalities per crash:**
  - **Mean 22.4**, **median 11** and std 35.1. The mean is double the median because a few huge crashes pull it up.
  - **Skewness 4.6** (strongly right-skewed): most crashes have few deaths, with a long tail up to 583.
  - **Kurtosis 36.5:** extreme outliers, far more than a normal distribution would have. For a "typical crash" the **median (11)** is the better number.
- **Survival rate per crash:**
  - **Mean 0.18**, **median 0**, std 0.31.
  - **Skewness 1.55:** the values pile up at 0, because in **63.9% of crashes nobody survived** and in only **1.4%** everybody survived.
  - The distribution is not bell-shaped at all, so the medians and the rank test (Kruskal–Wallis) are useful checks next to ANOVA and the T-tests.

### Hypothesis test 1: Did the average number of fatalities per crash change between decades?
- **H0:** the average fatalities per crash is the same in every decade.
- **H1:** at least one decade is different.

We use the decades from the 1930s to the 2010s (the earlier decades and the 2020s have few crashes) and a one-way **ANOVA** (`f_oneway`).

In [ ]:
decades = list(range(1930, 2020, 10))
# 1930, 1940, ..., 2010
dec_groups = [crashes.loc[crashes['Decade'] == d, 'Fatalities'] for d in decades]
# the fatalities of every decade, as a list of groups
f_value, p_value = stats.f_oneway(*dec_groups)
# one-way ANOVA
print('F-value:', round(f_value, 3), '| P-value:', f'{p_value:.2e}')
# the result
kw_stat, kw_p = stats.kruskal(*dec_groups)
# Kruskal-Wallis: the same question on ranks, safer for skewed data
print('Kruskal-Wallis P-value:', f'{kw_p:.2e}')
# check that both tests agree
print(crashes[crashes['Decade'].isin(decades)].groupby('Decade')['Fatalities'].agg(['size', 'mean', 'median']).round(1))
# crashes, mean and median fatalities per decade

### Hypothesis test 2: Do military and civil crashes have a different number of fatalities?
- **H0:** the average fatalities per crash is the same for military and civil crashes.
- **H1:** they are different.

We use an independent **T-test** (`ttest_ind`) with `equal_var=False` (Welch's version, because the two groups have very different sizes), and the critical value from `t.ppf`.

In [ ]:
mil = crashes.loc[crashes['Type'] == 'Military', 'Fatalities']
# fatalities in military crashes
civ = crashes.loc[crashes['Type'] == 'Civil', 'Fatalities']
# fatalities in civil crashes
t_stat, p_t = stats.ttest_ind(mil, civ, equal_var=False)
# Welch's T-test
crit = stats.t.ppf(1 - 0.05 / 2, len(mil) + len(civ) - 2)
# critical value for alpha = 0.05 (two-sided)
print('Military mean:', round(mil.mean(), 2), '| Civil mean:', round(civ.mean(), 2))
# the two averages
print('T-statistic:', round(t_stat, 3), '| critical value:', round(crit, 3), '| P-value:', f'{p_t:.2e}')
# the result
print('Survival rate - Military:', round(crashes.loc[crashes['Type'] == 'Military', 'Survival rate'].mean(), 3),
      '| Civil:', round(crashes.loc[crashes['Type'] == 'Civil', 'Survival rate'].mean(), 3))
# average survival rate of each type

### Hypothesis test 3: Is the survival rate higher in recent crashes?
We compare the survival rate of crashes **before 1980** with crashes **from 1980 on** (T-test).

In [ ]:
old = crashes.loc[crashes['Year'] < 1980, 'Survival rate'].dropna()
# survival rates before 1980
new = crashes.loc[crashes['Year'] >= 1980, 'Survival rate'].dropna()
# survival rates from 1980
t_s, p_s = stats.ttest_ind(new, old, equal_var=False)
# Welch's T-test: new minus old
print('Before 1980:', round(old.mean(), 3), '| From 1980:', round(new.mean(), 3))
# the two average survival rates
print('T-statistic:', round(t_s, 3), '| P-value:', f'{p_s:.2e}')
# positive t = higher survival in the recent period

**Interpretation of the tests:**
1. **Fatalities by decade (1930s–2010s):** **ANOVA F = 26.0, p ≈ 1e-39**; the Kruskal–Wallis test agrees (p ≈ 2e-75). **We reject H0:** the average deaths per crash really changed between decades.
   - It rose from **5.7 (1930s)** to a peak of **32.3 (1970s)**, then fell to 22–27.
   - Interesting detail: the **median** is highest in the 1960s (19) and falls to 8–10 after 1990. Since then most crashes are small, but the rare big-jet crashes keep the mean high.
2. **Military vs civil:** military crashes average **26.5** deaths and civil crashes **21.7**. **t = 4.49**, which is above the critical value of 1.96, and **p ≈ 7.7e-06**, so **we reject H0**: military crashes are deadlier, and their survival rate is also much lower (6.5% vs 20.2%).
3. **Survival before vs after 1980:** the average survival rate rose from **0.170 to 0.199**. **t = 3.15** and **p = 0.0016**, so the difference is **significant**: crashes became a little more survivable. The improvement is small (about 3 percentage points), though; even recently most crashes have no survivors.

---
# 4. Visualization

In [ ]:
plt.figure(figsize=(12, 5))
# a new wide chart
plt.plot(per_year.index, per_year.values, color='steelblue', label='Crashes per year')
# the number of crashes each year
plt.plot(per_year.index, per_year.rolling(5, center=True).mean(), color='red', label='5-year average')
# a smoothed line (average of 5 years) to see the trend
plt.title('Airplane crashes per year, 1908-2023')
# chart title
plt.xlabel('Year')
# label of the x axis
plt.ylabel('Number of crashes')
# label of the y axis
plt.legend()
# show the labels box
plt.show()
# show the chart

In [ ]:
deaths_year = crashes.groupby('Year')['Fatalities'].sum()
# total deaths each year
plt.figure(figsize=(12, 5))
# a new wide chart
plt.bar(deaths_year.index, deaths_year.values, color='darkred')
# one bar per year
plt.title('Fatalities per year')
# chart title
plt.xlabel('Year')
# label of the x axis
plt.ylabel('Fatalities')
# label of the y axis
plt.show()
# show the chart

In [ ]:
top_countries = crashes['Country'].value_counts().head(15)
# the 15 countries with the most crashes
plt.figure(figsize=(10, 6))
# a new chart
sns.barplot(x=top_countries.values, y=top_countries.index, color='teal')
# horizontal bars: one per country
plt.title('Top 15 countries by number of crashes')
# chart title
plt.xlabel('Number of crashes')
# label of the x axis
plt.ylabel('')
# no label needed on the y axis
plt.show()
# show the chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# two charts side by side
sns.histplot(crashes['Fatalities'], bins=60, ax=axes[0], color='darkred')
# histogram of fatalities per crash
axes[0].set_title('Fatalities per crash')
# title of the left chart
axes[0].set_xlabel('Fatalities')
# label of its x axis
sns.histplot(survival, bins=20, ax=axes[1], color='seagreen')
# histogram of the survival rate
axes[1].set_title('Survival rate per crash')
# title of the right chart
axes[1].set_xlabel('Survival rate (0 = nobody survived, 1 = everybody survived)')
# label of its x axis
plt.show()
# show both charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# two charts side by side
plot_data = crashes[crashes['Decade'].isin(decades)]
# only the decades used in the ANOVA
sns.barplot(data=plot_data, x='Decade', y='Fatalities', ax=axes[0], color='orange')
# average fatalities per decade; the black lines are 95% confidence intervals
axes[0].set_title(f'Average fatalities per crash by decade (ANOVA p = {p_value:.1e})')
# title with the test result
sns.boxplot(data=crashes, x='Type', y='Survival rate', ax=axes[1])
# survival rate distribution for military and civil crashes
axes[1].set_title('Survival rate: Military vs Civil')
# title of the right chart
plt.show()
# show both charts

**What the charts show:**
- **Crashes per year:** a fast rise until the late 1940s (peak in 1946), a long plateau from 1950 to 2000, and a **sharp decline after 2000** down to 4 crashes in 2023. Part of the late drop may come from the dataset recording fewer small crashes in recent years, but the drop is large and steady.
- **Fatalities per year:** the deadliest years are the **1970s–1990s** (the worst year is **1972, with 2,796 deaths**). Planes were big but not yet as safe as today. Deaths fall after 2000 along with the crashes.
- **Top countries:** the **USA has by far the most crashes** (1,005), about 3 times as many as Russia (324). This reflects how much flying happens in the USA, not that its flights are more dangerous.
- **Histograms:** the fatalities chart is a tall bar near 0 with a long tail to the right (skewness 4.6). The survival rate chart is a huge bar at 0 (64% of crashes) with a small bump at 1.
- **By decade:** the bar chart shows the rise to the 1970s peak and the later fall (ANOVA p ≈ 1e-39). The 2010s bar has a wide confidence interval because there are fewer crashes.
- **Military vs civil boxplot:** the military box is flat at 0 (most military crashes had no survivors). The civil box reaches up to about 0.33.

---
# 5. Insight and Report

## Report: Airplane Crashes and Fatalities, 1908–2023

**The data.** 4,998 recorded crashes (4,980 after removing 18 with unknown numbers aboard). Together they had **155,356 people aboard** and **111,561 deaths** aboard.

**Key findings**
1. **Crashes have become much rarer.** Crashes peaked after World War II (**87 in 1946**) and stayed at 50–80 per year until about 2000. Since then they have **fallen steadily to 4 in 2023**: since 1970 the correlation between year and crash count is r = −0.77 (p ≈ 8e-12).
2. **Crashes are usually all-or-nothing.** In **64% of crashes nobody survived**. Overall, 28% of the people aboard survived, but the average crash had a survival rate of only 18%.
3. **Deaths per crash are very skewed.** The median is **11** but the mean is **22.4** (skewness 4.6, kurtosis 36.5). A few disasters dominate the totals, for example **Tenerife 1977, with 583 deaths**.
4. **The size of crashes changed by decade** (ANOVA F = 26.0, p ≈ 1e-39). Bigger planes raised the average from 5.7 deaths (1930s) to **32.3 (1970s)**. After that the typical (median) crash got smaller.
5. **Military crashes are deadlier** (26.5 vs 21.7 deaths, t = 4.49, p ≈ 8e-06) and much less survivable (6.5% vs 20.2%).
6. **Survival improved slightly after 1980** (0.170 → 0.199, p = 0.0016).
7. **Where:** the **USA (1,005)**, Russia/USSR (324), Brazil, Colombia and the UK have the most crashes. These are mostly countries with a lot of flying or with difficult terrain.

**Anomalies and data issues.**
- The 9/11 ground victims (2,750) are recorded on both planes.
- Locations are free text, so the country list is messy.
- "Military" is detected only from the operator name.
- Recent years may be less complete in this dataset.

**How I used the tools**
- **Pandas:**
  - loading the file (`read_csv` with latin-1 encoding)
  - `to_datetime` and the Year/Decade columns
  - `fillna`/`dropna` to clean
  - `str.split` to get the country
  - `groupby`, `value_counts`, `nlargest` and `describe` for all the summaries
- **NumPy:**
  - `np.where` for the survival rate and the military/civil type
  - `np.median`
  - turning the yearly counts into arrays for the trend analysis
- **SciPy:**
  - `tmean`, `tstd`, `skew` and `kurtosis` to describe the distributions
  - `f_oneway` (ANOVA) and `kruskal` to compare decades
  - `ttest_ind` with `t.ppf` for military vs civil and before vs after 1980
  - `pearsonr` and `linregress` for the trend over time
- **Matplotlib / Seaborn:** time series, bar charts, histograms and boxplots.

**Insight.** Flying has become far safer: there are fewer crashes each year, and survival is slowly improving. But when a crash happens, it is still usually fatal for everyone aboard, and a few large accidents account for a big share of all deaths.